# Full-data naive offline CRL on the ROCKFALL AntMaze benchmark (Colab GPU)

Faithful offline contrastive-RL recipe (bc 0.05, twin-min, alpha 0, batch 1024,
repr 16, hidden (1024,1024)) on the FROZEN rockfall datasets. Learner sees only
obs/act (no mask/route/mode). Eval runs in `offline_ant_umaze_rockfall`.

One notebook, three datasets (set `DATASET` in the config cell, run once each):
- `full`         -> main naive run, >=300k updates (extend if still climbing);
- `center_only`  -> ORACLE: only center-route episodes (is center learnable?);
- `reweight_c50` -> ORACLE: center episodes upweighted to ~51% of transitions.

Oracle sets use privileged sidecar route labels -- diagnostic only, NOT fair
baselines. NPZs pull from Drive (>100MB); upload per
`artifacts/rockfall_dataset/DATASETS.md`. Rich behavioural metrics run on the
WORKSTATION after download via `scripts/diagnose_naive_rockfall.py`.


## 1. Configuration -- single source of truth

In [ ]:
# 1. Configuration. Everything tunable lives here.
import os

# --- which dataset to train on (run the notebook once per choice) -------------
DATASET = 'full'    # 'full' | 'center_only' | 'reweight_c50'
DATASETS = {
    'full':         {'file': 'antmaze_rockfall_full.npz', 'sub': 'full',
                     'sha': 'd9722c91422c3b98eeb1a76e351352fc83a0b2086784c336b3f0928f2110c7f0',
                     'steps': 300_000, 'tag': 'full'},
    'center_only':  {'file': 'antmaze_rockfall_full_center_only.npz', 'sub': 'oracle',
                     'sha': '7deba09d4345848dd41214a654c69f2d17ec166e1b3071125223f484879a0559',
                     'steps': 200_000, 'tag': 'centeronly'},
    'reweight_c50': {'file': 'antmaze_rockfall_full_reweight_c50.npz', 'sub': 'oracle',
                     'sha': '47cf12eba347dcec12b90c544826ed681c3adc5a5493eaf1d810452dc0df4833',
                     'steps': 300_000, 'tag': 'reweightc50'},
}
SEED        = 0
RESUME      = False           # True after a Colab disconnect (restores latest.pkl)
REQUIRE_GPU = True
_D = DATASETS[DATASET]
MAX_UPDATES = _D['steps']     # full: >=300k; extend if the eval curve still climbs

# --- repo (pinned to the full-data pipeline commit) ---------------------------
REPO     = 'contrastive_rl'
REPO_URL = 'https://github.com/tingrui-huang/contrastive_rl.git'
BRANCH   = 'main'
COMMIT   = '24b089c'          # full-data rockfall pipeline (1400-ep + R1-R13 audit)

# --- dataset: pulled from Drive (NPZ > 100MB; see DATASETS.md) -----------------
ENV_NAME           = 'offline_ant_umaze_rockfall'
DATASET_SHA256     = _D['sha']
DATASET_DRIVE_DIR  = '/content/drive/MyDrive/contrastive_rl_datasets/rockfall_dataset'
DATASET_DRIVE_PATH = f'{DATASET_DRIVE_DIR}/{_D["sub"]}/{_D["file"]}'
LOCAL_DATASET_PATH = f'/content/data/{_D["file"]}'

# --- run dirs -----------------------------------------------------------------
RUN_ID        = f'naive_rockfall_{_D["tag"]}_s{SEED}_{MAX_UPDATES//1000}k'
LOCAL_RUN_DIR = f'/content/runs/{RUN_ID}'
RUN_DRIVE_DIR = f'/content/drive/MyDrive/contrastive_rl_runs/{RUN_ID}'
print(RUN_ID, '| dataset', DATASET, '| steps', MAX_UPDATES)


## 2. Mount Google Drive; create dirs

In [ ]:
# 2. Mount Drive; make local scratch + Drive run trees + local data dir.
from google.colab import drive
drive.mount('/content/drive')
for base in (LOCAL_RUN_DIR, RUN_DRIVE_DIR, '/content/data'):
    os.makedirs(base, exist_ok=True)
print('local scratch:', LOCAL_RUN_DIR)
print('Drive run dir:', RUN_DRIVE_DIR)
!df -h /content | tail -1


## 3. Clone/checkout the repo (refuses to overwrite uncommitted work)

In [ ]:
# 3. Clone if absent; otherwise fetch + checkout at COMMIT (or origin/BRANCH).
import subprocess, sys
os.chdir('/content')
if not os.path.exists(REPO):
    !git clone $REPO_URL $REPO
os.chdir(REPO)
dirty = subprocess.run(['git','status','--porcelain'], capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError('repo has uncommitted changes -- refusing to checkout over them:\n' + dirty)
!git fetch -q origin
ref = COMMIT if COMMIT else f'origin/{BRANCH}'
!git checkout -q $ref
!git log -1 --oneline
for req in ('crl/offline_audit.py','crl/d4rl_ant.py','crl/rockfall_ant.py','crl/train.py',
            'scripts/naive_rockfall_crl.py','scripts/verify_offline_d4rl.py',
            'scripts/naive_rockfall_crl.py',
            'scripts/diagnose_naive_rockfall.py'):
    if not os.path.exists(req):
        raise RuntimeError(f'{req} missing -- push main from the workstation and rerun.')
print('rockfall pipeline files present -- checkout OK')

## 4. Dependencies (preserve Colab's preinstalled GPU JAX -- do not alter)

In [ ]:
# 4. Install deps WITHOUT disturbing Colab's GPU JAX (pin jax/jaxlib/numpy).
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'
import jax, jaxlib, numpy
hold = [f'jax=={jax.__version__}', f'jaxlib=={jaxlib.__version__}', f'numpy=={numpy.__version__}']
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
print('Colab JAX', jax.__version__, '| devices:', jax.devices())
pip('--no-deps', 'dm-haiku', 'optax', 'chex')
pip('jmp', 'tabulate', 'toolz', 'etils', 'tensorboardX', 'mujoco', 'imageio', 'imageio-ffmpeg', *hold)
print('post-install JAX', jax.__version__, '| devices:', jax.devices())

## 5. GPU / environment verification

In [ ]:
# 5. Require an accelerator; record env meta to Drive.
import hashlib, json, platform, mujoco
os.chdir('/content/'+REPO)
commit = subprocess.run(['git','rev-parse','HEAD'], capture_output=True, text=True).stdout.strip()
meta = {'run_id': RUN_ID, 'python': platform.python_version(), 'jax': jax.__version__,
        'backend': jax.default_backend(), 'devices':[str(d) for d in jax.devices()],
        'mujoco': mujoco.__version__, 'git_commit': commit, 'env': ENV_NAME}
if REQUIRE_GPU and jax.default_backend() == 'cpu':
    raise RuntimeError('no accelerator -- Runtime > Change runtime type > GPU')
!nvidia-smi -L || true
json.dump(meta, open(f'{RUN_DRIVE_DIR}/meta_env.json','w'), indent=2)
print(json.dumps(meta, indent=1))

## 6. Dataset: pull the fixed NPZ from Drive -> verify SHA

Upload the three NPZs to the Drive path in the config cell first (see `artifacts/rockfall_dataset/DATASETS.md`).


In [ ]:
# 6. Fetch the fixed rockfall npz from Drive -> /content/data; verify SHA.
import shutil
def sha256(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
if not os.path.exists(DATASET_DRIVE_PATH):
    raise FileNotFoundError('dataset missing on Drive: ' + DATASET_DRIVE_PATH
        + '  -- upload it (see artifacts/rockfall_dataset/DATASETS.md)')
if not os.path.exists(LOCAL_DATASET_PATH):
    shutil.copy2(DATASET_DRIVE_PATH, LOCAL_DATASET_PATH)
got = sha256(LOCAL_DATASET_PATH)
if got != DATASET_SHA256:
    raise SystemExit('dataset sha mismatch: got ' + got + ' exp ' + DATASET_SHA256)
print('dataset OK:', LOCAL_DATASET_PATH, '|', got[:16], '...')


## 7. Pre-training offline audit -- must PASS before any training

In [ ]:
# 7. Static offline gates (G1-G8) on the REAL rockfall dataset.
os.chdir('/content/'+REPO); sys.path.insert(0, '/content/'+REPO)
from crl import offline_audit
from crl.config import Config
from crl import envs as envs_mod
_c = Config(env_name=ENV_NAME, offline_dataset=LOCAL_DATASET_PATH)
envs_mod.make_env(ENV_NAME, _c, seed=0)   # fills obs/goal/action dims
passed, gates, rep = offline_audit.run_static_audit(LOCAL_DATASET_PATH, _c)
print('offline_audit gates:', {g:('PASS' if ok else 'FAIL') for g,ok in gates.items()})
fp = rep['fingerprint']
print(f"  sha256={fp['sha256'][:16]}  eps={fp['n_episodes']}  trans={fp['n_transitions']}  obs={fp['obs_shape']}")
assert passed, 'offline_audit FAILED -- see gates'

## 8. (resume) restore latest.pkl from Drive

In [ ]:
# 8. If RESUME, pull the last checkpoint from Drive into local scratch.
import glob, shutil, os
if RESUME:
    for f in glob.glob(f'{RUN_DRIVE_DIR}/*.pkl') + [f'{RUN_DRIVE_DIR}/metrics.json',
                                                    f'{RUN_DRIVE_DIR}/offline_dataset.sha256']:
        if os.path.exists(f): shutil.copy2(f, f'{LOCAL_RUN_DIR}/{os.path.basename(f)}')
    print('restored:', sorted(os.path.basename(p) for p in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl')))
else:
    print('fresh run (RESUME=False)')

## 9. TensorBoard

In [ ]:
tb = f'{LOCAL_RUN_DIR}/tb'
%load_ext tensorboard
%tensorboard --logdir $tb

## 10. Launch training (live stream)

Trains to Drive-mirrored local scratch. Eval every 10k in the rockfall env.

In [ ]:
# 10. Launch naive offline CRL on the rockfall dataset (LIVE streaming).
os.chdir('/content/'+REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', 'scripts/naive_rockfall_crl.py',
       '--npz', LOCAL_DATASET_PATH, '--steps', str(MAX_UPDATES),
       '--seed', str(SEED), '--ckpt-dir', LOCAL_RUN_DIR]
if RESUME: cmd.append('--resume')
print(' '.join(cmd)); print('-'*70)
proc = subprocess.Popen(cmd, env={**os.environ}, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\n[driver exited {rc}]')

## 11. Mirror checkpoints to Drive

In [ ]:
# 11. Copy checkpoints + metrics to the persistent Drive dir.
import glob, shutil, os, json
n = 0
for f in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl') + glob.glob(f'{LOCAL_RUN_DIR}/*.json') + glob.glob(f'{LOCAL_RUN_DIR}/*.sha256'):
    shutil.copy2(f, f'{RUN_DRIVE_DIR}/{os.path.basename(f)}'); n += 1
print(f'mirrored {n} files to {RUN_DRIVE_DIR}')
mp = f'{LOCAL_RUN_DIR}/metrics.json'
if os.path.exists(mp):
    for e in json.load(open(mp))[-8:]:
        print({k: e.get(k) for k in ('step','success','min_dist','final_dist')})

## 12. Done -- download for the workstation diagnosis

Training + checkpoints are mirrored to Drive. Behavioural characterisation
(route distribution, trigger-avoidance scan, paired mask-flip) runs on the
WORKSTATION: download `best.pkl` (+ `metrics.json`) from the Drive run dir into
`artifacts/naive_rockfall_crl/` and run
`python scripts/diagnose_naive_rockfall.py --ckpt artifacts/naive_rockfall_crl/best.pkl`.


In [ ]:
# 12. Done -- summary; run the behavioural diagnosis on the WORKSTATION.
import json
mp = f'{LOCAL_RUN_DIR}/metrics.json'
if os.path.exists(mp):
    for e in json.load(open(mp))[-8:]:
        print({k: e.get(k) for k in ('step','success','min_dist','final_dist') if k in e})
print('checkpoints mirrored to:', RUN_DRIVE_DIR)
print('download best.pkl + metrics.json to artifacts/<RUN_ID>/, then on the workstation:')
print('  python scripts/diagnose_naive_rockfall.py --ckpt artifacts/<RUN_ID>/best.pkl')
